#### L01 - Paradigmas
Rafael Campideli Hoyos - 175100

- Base da estruturação do grafo no scheme (este trecho dde código foi compartilhado entre os membros da equipe TAGAS):
    - OBS: na conceitualização da linguagem TAGAS ainda não existe definição de PESOS para arestas e, por isso mesmo, não implementaremos por enquanto, mas colocamos aqui na estrutura base só pra ter uma ideia de como poderia ficar

In [163]:
; base do grafo (compartilhada entre os membros do grupo do projeto TAGAS)

(define graph-schema
    '((directed . 0) (weighted . 1) (direct-loop . 2) (body . 3))
)

(define (getp g p)
    (list-ref g (cdr (assq p graph-schema)))
)

(define (replace-by-i i value l)
    (if (= i 0)
        (cons value (cdr l))
        (cons (car l) (replace-by-i (- i 1) value (cdr l)))
    )
)

(define (setp g p value)
    (replace-by-i (cdr (assq p graph-schema)) value g)
)

- Implementação da primitiva de adicionar vértices e arestas:

In [164]:
(define (add-vertex grafo v)
    (let ((body (getp grafo 'body)))
        (if (assq v body)
            grafo
            (setp grafo 'body (cons `(,v . ()) body))
        )
    )
)

(define (add-vertexs grafo . vertexs)
    (if (null? vertexs)
        grafo
        (apply add-vertexs `(,(add-vertex grafo (car vertexs)) ,@(cdr vertexs)))
    )
)

(define (replace-by-key key value l)
    (cond
        ((null? l)
            l)
        ((eq? (car (car l)) key)
            (cons `(,key . ,value) (cdr l)))
        (else (cons (car l)
            (replace-by-key key value (cdr l))))
    )
)

; apenas adiciona v2 na lista de adjacências de v1, sem se preocupar com o tipo do grafo
; e sem checar a existência de v1 e v2 antes (dá erro se v1 não existir)
(define (add-adjacency grafo v1 v2)
    (let ((body (getp grafo 'body)))
        (let ((v1-edges (cdr (assq v1 body))))
            (if (memq v2 v1-edges)
                grafo
                (setp grafo 'body (replace-by-key v1 (cons v2 v1-edges) body))
            )
        )
    )
)

; adiciona a aresta entre v1 e v2, checando o tipo do grafo
; se for não direcionado adiciona a adjacência v1-v2 e v2-v1
(define (add-edge grafo v1 v2)
    (let ((body (getp grafo 'body)))
        (if (assq v1 body)
            (if (assq v2 body)
                (if (getp grafo 'directed)
                    (add-adjacency grafo v1 v2)
                    (add-adjacency (add-adjacency grafo v1 v2) v2 v1)
                )
                (add-edge (add-vertex grafo v2) v1 v2)
            )
            (add-edge (add-vertex grafo v1) v1 v2)
        )
    )
)

; Primitiva do "define" para adicionar arestas na linguagem tagas: define A>B B>C D>A
; se limitando somente pares
; exemplo: (add-edges grafo (A B) (B C) (C A))
(define (add-edges grafo . edges)
    (if (null? edges)
        grafo
                ;   (add-edges grafo . (cdr edges))
        (apply add-edges `(,(apply add-edge (list grafo (car (car edges)) (cadr (car edges)))) ,@(cdr edges)))
    )
)

- Exemplo / teste de uso da implementação:

In [166]:
; uso
; (directed   (not) weighted   (not) direct-loop   body)
(let ((GRAFO '(#t #f #f ())))

    ; inicialmente planejava usar macros aqui, mas estou usando o interpretador Calysto Scheme que não suporta macros da maneira que vimos em sala
    (define (print-body)
        (display (getp GRAFO 'body))
    )
    (define (add-vertex! V)
        (set! GRAFO (add-vertex GRAFO V))
    )
    (define (add-vertexs! . vertexs)
        (set! GRAFO (apply add-vertexs `(,GRAFO ,@vertexs)))
    )
    (define (add-edge! V1 V2)
        (set! GRAFO (add-edge GRAFO V1 V2))
    )
    (define (add-edges! . edges)
        (set! GRAFO (apply add-edges `(,GRAFO ,@edges)))
    )

    (print-body)
    
    (newline)
    ; define B
    (add-vertex! 'B)
    (print-body)
    
    (newline)
    ; define M N
    (add-vertexs! 'M 'N)
    (print-body)

    (newline)
    ; define B>A
    (add-edge! 'B 'A)
    (print-body)

    (newline)
    ; define B>M
    (add-edge! 'B 'M)
    (print-body)

    (newline)
    ; define R>G Y>F Y>K
    (add-edges! '(R G) '(Y F) '(Y K))
    (print-body)
)

; mesma ideia, só que trocando o modo para NÃO direcionado
; ((not) directed   (not) weighted   (not) direct-loop   body)
(let ((GRAFO '(#f #f #f ())))

    ; inicialmente planejava usar macros aqui, mas estou usando o interpretador Calysto Scheme que não suporta macros da maneira que vimos em sala
    (define (print-body)
        (display (getp GRAFO 'body))
    )
    (define (add-vertex! V)
        (set! GRAFO (add-vertex GRAFO V))
    )
    (define (add-vertexs! . vertexs)
        (set! GRAFO (apply add-vertexs `(,GRAFO ,@vertexs)))
    )
    (define (add-edge! V1 V2)
        (set! GRAFO (add-edge GRAFO V1 V2))
    )
    (define (add-edges! . edges)
        (set! GRAFO (apply add-edges `(,GRAFO ,@edges)))
    )

    (newline)
    (newline)
    (print-body)
    
    (newline)
    ; define B
    (add-vertex! 'B)
    (print-body)
    
    (newline)
    ; define M N
    (add-vertexs! 'M 'N)
    (print-body)

    (newline)
    ; define B>A
    (add-edge! 'B 'A)
    (print-body)

    (newline)
    ; define B>M
    (add-edge! 'B 'M)
    (print-body)

    (newline)
    ; define R>G Y>F Y>K
    (add-edges! '(R G) '(Y F) '(Y K))
    (print-body)
)

()
((B))
((N) (M) (B))
((A) (N) (M) (B A))
((A) (N) (M) (B M A))
((K) (F) (Y K F) (G) (R G) (A) (N) (M) (B M A))

()
((B))
((N) (M) (B))
((A B) (N) (M) (B A))
((A B) (N) (M B) (B M A))
((K Y) (F Y) (Y K F) (G R) (R G) (A B) (N) (M B) (B M A))